**#Step 3. Validation#**

We performed three independent validation strategies to assess the robustness of the identified markers:

**Train/test split on the original osteoporosis dataset** – using k‑mer features extracted by MetaFX.

**Train/test split on the original osteoporosis dataset** – using taxonomic profiles from Kraken2.

**External validation** – applying the k‑mer‑based model to an independent arthritis cohort (colleague's data used in independent investigation (LINK)).

In all approaches, we applied feature filtering: only features (k‑mers or taxa) present in at least 5% of the training samples were retained, which improved prediction stability.

**Substep 3.1 Train/test split with MetaFX (k‑mer features)**

The original 56 samples (37 normal, 20 low) were split into training (44) and test (12) sets, preserving class proportions. The test set contained 4 low‑BMD and 8 normal‑BMD samples.

*(lists of test and train samples are available in linked zenodo repo as test_BMD_samples.txt and train_BMD_samples.tsv)*

In [ ]:
#!/usr/bin/env python
# split_train_test.py
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("sample_categories_bmd.txt", sep="\t", header=None, names=["sample", "category"])
train, test = train_test_split(df, test_size=12/56, random_state=42, stratify=df["category"])
train.to_csv("train_samples.txt", sep="\t", index=False, header=False)
test.to_csv("test_samples.txt", sep="\t", index=False, header=False)
print(f"Train: {len(train)}, Test: {len(test)}")

**Substep 3.1.1** Prepare kmers for training (copy from old directory)

In [ ]:
#!/bin/bash
#SBATCH -J copy_kmers
#SBATCH -n 1
#SBATCH --mem=8G
#SBATCH -t 0:30:00

cd /mnt/tank/scratch/ris/SRR_files/validate

# Create directory for kmers
mkdir -p wd_unique_train/kmers/kmers

# Copy kmers for all training samples (names with _r1)
cut -f1 train_samples.txt | sed 's/_r1.fastq.gz//' | while read name; do
    src="/mnt/tank/scratch/ris/SRR_files/wd_unique_bmd/kmers/kmers/${name}_r1.kmers.bin"
    dst="wd_unique_train/kmers/kmers/${name}_r1.kmers.bin"
    if [ -f "$src" ]; then
        cp -v "$src" "$dst"
    else
        echo "Skipped: $name"
    fi
done

# Create symlinks without _r1 (if needed)
cd wd_unique_train/kmers/kmers
for f in *_r1.kmers.bin; do
    ln -sf "$f" "${f/_r1./.}"
done

**Substep 3.1.2** Run metafx unique on training set

In [ ]:
#!/bin/bash
# run_unique_train.sbatch
#SBATCH -J unique_train
#SBATCH -n 1
#SBATCH --cpus-per-task=8
#SBATCH --mem=128G
#SBATCH -t 12:00:00
#SBATCH --output=unique_train_%j.out
#SBATCH --error=unique_train_%j.err

cd /mnt/tank/scratch/ris/SRR_files/validate
source /nfs/home/ris/miniforge3/etc/profile.d/mamba.sh
mamba activate snakemake

# Remove old temporary directories (for restart)
rm -rf wd_unique_train/unique_kmers_* wd_unique_train/components_* wd_unique_train/contigs_* wd_unique_train/features_*
rm -f wd_unique_train/feature_table.tsv

metafx unique -t 8 -m 128G -w wd_unique_train -k 31 -i train_samples.txt --kmers-dir wd_unique_train/kmers/kmers


**Substep 3.1.3** Compute features for test set

In [ ]:
#!/bin/bash
# calc_features_test.sbatch
#SBATCH -J calc_test
#SBATCH -n 1
#SBATCH --cpus-per-task=8
#SBATCH --mem=64G
#SBATCH -t 6:00:00
#SBATCH --output=calc_test_%j.out
#SBATCH --error=calc_test_%j.err

cd /mnt/tank/scratch/ris/SRR_files/validate
source /nfs/home/ris/miniforge3/etc/profile.d/mamba.sh
mamba activate snakemake

# Create a list of absolute paths for test samples
awk '{print "/mnt/tank/scratch/ris/SRR_files/" $1}' test_samples.txt > test_abs.txt

metafx calc_features -t 8 -m 16G -w wd_calc_features_test -k 31 -d wd_unique_train -i $(cat test_abs.txt)

**Substep 3.1.4** Preprocess feature tables (filter low‑prevalence k‑mers, normalise) 

In [ ]:
#!/bin/bash
# preprocess_metafx_final.sbatch
#SBATCH -J preproc_metafx
#SBATCH -n 1
#SBATCH --cpus-per-task=2
#SBATCH --mem=8G
#SBATCH -t 0:30:00
#SBATCH --output=preproc_metafx.out
#SBATCH --error=preproc_metafx.err

cd /mnt/tank/scratch/ris/SRR_files/validate
source /nfs/home/ris/miniforge3/etc/profile.d/mamba.sh
mamba activate snakemake

python - <<'EOF'
import pandas as pd

def clean_names(s):
    return s.str.replace("_r1", "", regex=False).str.replace(".fastq.gz", "", regex=False)

# Train
train_raw = pd.read_csv("wd_unique_train/feature_table.tsv", sep="\t", index_col=0).T
train_raw.index = clean_names(train_raw.index)
train_labels = pd.read_csv("train_samples.txt", sep="\t", header=None, names=["sample", "category"])
train_labels["sample"] = clean_names(train_labels["sample"])
common = train_raw.index.intersection(train_labels["sample"])
X_train = train_raw.loc[common]
y_train = train_labels.set_index("sample").loc[common]

# Filter features (>=5% of train samples)
min_frac = 0.05
keep = (X_train > 0).sum(axis=0) >= (len(X_train) * min_frac)
keep_cols = keep[keep].index
X_train_filt = X_train[keep_cols]
X_train_norm = X_train_filt.div(X_train_filt.sum(axis=1), axis=0).fillna(0)

# Test
test_raw = pd.read_csv("wd_calc_features_test/feature_table.tsv", sep="\t", index_col=0).T
test_raw.index = clean_names(test_raw.index)
X_test = test_raw[keep_cols]
X_test_norm = X_test.div(X_test.sum(axis=1), axis=0).fillna(0)

# Save cleaned tables (transposed back to features × samples)
X_train_norm.T.to_csv("feature_table_train_clean.tsv", sep="\t")
X_test_norm.T.to_csv("feature_table_test_clean.tsv", sep="\t")

# Save labels for fit
y_train.to_csv("train_categories_clean.tsv", sep="\t", header=False)
print("Preprocessing done. Files saved.")
EOF

**Subset 3.1.5** Train model using metafx fit
 

In [ ]:
metafx fit -w wd_fit -f feature_table_train_clean.tsv -i train_categories_clean.tsv --name rf_model

**Subset 3.1.6** Predict on test set using metafx predict

In [ ]:
metafx predict -w wd_predict -f feature_table_test_clean.tsv --model wd_fit/rf_model.joblib --name predictions

**Subset 3.1.7** Evaluate predictions 

In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report

pred = pd.read_csv("wd_predict/predictions.tsv", sep="\t", header=None, names=["sample", "predicted"])
true = pd.read_csv("test_samples.txt", sep="\t", header=None, names=["sample", "true"])
true["sample"] = true["sample"].str.replace("_r1.fastq.gz", "", regex=False)
merged = pd.merge(pred, true, on="sample")
acc = accuracy_score(merged["true"], merged["predicted"])
print(f"Test accuracy: {acc:.3f}")
print(classification_report(merged["true"], merged["predicted"]))

**Substep 3.1.8** Extract top‑20 contigs from Random Forest model

In [ ]:
import pandas as pd
import joblib
from Bio import SeqIO
import os

# Load trained model
model = joblib.load("wd_fit/rf_model.joblib")
importances = model.feature_importances_

# Load feature names (from the cleaned training feature table)
# The feature table has features as rows, samples as columns.
feature_table = pd.read_csv("feature_table_train_clean.tsv", sep="\t", index_col=0)
feature_names = feature_table.index.tolist()

# Create importance DataFrame and sort
imp_df = pd.DataFrame({"feature": feature_names, "importance": importances})
imp_df = imp_df.sort_values("importance", ascending=False)
top20 = imp_df.head(20)

# Extract sequences for each top feature
out_fasta = "top20_contigs_metafx_preproc.fasta"
with open(out_fasta, "w") as out:
    for _, row in top20.iterrows():
        feat = row["feature"]
        # Features are named as "category_index", e.g., "low_305", "normal_613"
        try:
            category, idx_str = feat.split("_")
            idx = int(idx_str)
            # Path to the contig FASTA file for that category (produced by `metafx unique`)
            fasta_file = f"wd_unique_train/contigs_{category}/components.seq.fasta"
            if not os.path.exists(fasta_file):
                print(f"Warning: {fasta_file} not found, skipping {feat}")
                continue
            # Parse FASTA and extract the sequence at position 'idx'
            records = list(SeqIO.parse(fasta_file, "fasta"))
            if idx < len(records):
                rec = records[idx]
                # Modify header to include feature name and importance
                rec.id = f"{feat}_imp_{row['importance']:.6f}"
                rec.description = ""
                SeqIO.write(rec, out, "fasta")
            else:
                print(f"Index {idx} out of range for {category} (max {len(records)-1})")
        except Exception as e:
            print(f"Error parsing {feat}: {e}")

print(f"Top‑20 contigs saved to {out_fasta}")

**Substep 3.2 Train/test split with Kraken2 (taxonomic profiles)**

The same train/test split was applied to the relative abundance profiles obtained from Kraken2 classification

*Assumption: the file kraken_step1.csv is already exists (samples × taxa)* 

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Load Kraken data
df = pd.read_csv('kraken_species_reads.csv', index_col=0)
# Transpose: samples -> rows, taxa -> columns
df_T = df.T
print(f"Original table: {df_T.shape[0]} samples, {df_T.shape[1]} taxa")

# Original table: 57 samples, 16,873 taxa


**Substep 3.2.1** Training set (based on list)

In [ ]:
train_samples_raw = [
    "SRR25006915", "SRR25006904", "SRR25006916", "SRR25006888", "SRR25006874",
    "SRR25006889", "SRR25006912", "SRR25006894", "SRR25006905", "SRR25006869",
    "SRR25006920", "SRR25006881", "SRR25006922", "SRR25006876", "SRR25006880",
    "SRR25006891", "SRR25006883", "SRR25006878", "SRR25006892", "SRR25006896",
    "SRR25006923", "SRR25006907", "SRR25006903", "SRR25006910", "SRR25006875",
    "SRR25006900", "SRR25006890", "SRR25006882", "SRR25006917", "SRR25006913",
    "SRR25006871", "SRR25006886", "SRR25006877", "SRR25006924", "SRR25006897",
    "SRR25006879", "SRR25006921", "SRR25006870", "SRR25006898", "SRR25006867",
    "SRR25006925", "SRR25006873", "SRR25006872", "SRR25006908", "SRR25006895"
]

train_labels = [
    "normal", "normal", "low", "normal", "normal",
    "normal", "normal", "normal", "low", "low",
    "normal", "normal", "low", "normal", "normal",
    "low", "normal", "normal", "low", "normal",
    "normal", "normal", "normal", "normal", "low",
    "low", "normal", "normal", "low", "normal",
    "low", "normal", "normal", "normal", "normal",
    "low", "low", "low", "normal", "normal",
    "low", "normal", "low", "normal", "low"
]

# Ensure all samples exist in the data
train_samples = [s for s in train_samples_raw if s in df_T.index]
train_labels = train_labels[:len(train_samples)]


**Substep 3.2.2** Load true labels for test set

In [ ]:
test_true = {}
with open('test_BMD_samples.txt', 'r') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        parts = line.split()
        if len(parts) == 2:
            sample_name = parts[0].replace('_r1.fastq.gz', '').upper()
            label = parts[1]
            test_true[sample_name] = label
            

**Substep 3.2.3** Data processing

In [ ]:
# Feature preprocessing
# Remove taxa present in less than 5% of samples
min_samples_frac = 0.05
taxa_to_keep = (df_T > 0).sum(axis=0) >= len(df_T) * min_samples_frac
X_filtered = df_T.loc[:, taxa_to_keep]

# Normalization: relative abundances
X_norm = X_filtered.div(X_filtered.sum(axis=1), axis=0).fillna(0)

# Split into train / test
X_train = X_norm.loc[train_samples]
y_train = pd.Series(train_labels, index=train_samples)

# Test samples – all not in train
X_test = X_norm.loc[~X_norm.index.isin(train_samples)]
# Keep only samples with true labels
X_test = X_test.loc[X_test.index.isin(test_true.keys())]
y_test = pd.Series({s: test_true[s] for s in X_test.index})


**Substep 3.2.4** Model training

In [ ]:
model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_pred_prob = model.predict_proba(X_test)[:, 1]  # probability of class "low"

# Metrics
accuracy = accuracy_score(y_test, y_pred)
print(f"\nTest set accuracy: {accuracy:.3f}")
print("\nClassification report:")
print(classification_report(y_test, y_pred))
print("\nConfusion matrix:")
cm = confusion_matrix(y_test, y_pred, labels=['normal','low'])
print(cm)

# Visualize confusion matrix
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['normal','low'], yticklabels=['normal','low'])
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.savefig('Step3_kraken_confusion_matrix.png')
plt.show()


![Confusion matrix on Kraken2'](./images/BMD/Step3_kraken_confusion_matrix.png)

**Substep 3.2.5** Predictions results

In [ ]:
results = pd.DataFrame({
    'sample': X_test.index,
    'true_label': y_test,
    'predicted_label': y_pred,
    'probability_low': y_pred_prob
})
results.to_csv('test_predictions_with_true_labels.csv', index=False)
print("\nResults saved to 'test_predictions_with_true_labels.csv'")

# Top-10 important taxa
feature_importance = pd.DataFrame({
    'taxon': X_train.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop-10 most important taxa:")
print(feature_importance.head(10))


Top-10 most important taxa:

||taxon|  importance|
|----|------------------------------------------|------------|
|2856|   Clostridium saccharoperbutylacetonicum |   0.013596|
|2844|                      Clostridium gelidum |   0.011232|
|5746|            Macellibacteroides fermentans|    0.010464|
|6302|    Methylophaga nitratireducenticrescens|    0.009682|
|8184|                  Pedobacter sp. MW01-1-1|    0.009170|
|10766|                       Spinacia oleracea|    0.007887|
|4830|                    Heyndrickxia oleronia|    0.007867|
|2847|                     Clostridium kluyveri|    0.007701|
|4371|                  Gilliamella sp. ESL0443|    0.007326|
|7330|               Nocardioides campestrisoli|    0.007214|



**Substep 3.3 External validation on independent arthritis cohort**

We used the feature directory wd_unique_bmd (from the full osteoporosis dataset) to compute features for the arthritis samples, then predicted with the model trained (on all dataset) in Step2/Substep 2.1.5 

Arthritis dataset discription is available in (LINK)

**Substep 3.3.1** Compute features for arthritis data

Require *files_arth_filtered.txt* with paths to files; set permissions to 755 (rwxr-xr-x)

In [ ]:
#!/bin/bash
#SBATCH -J calc_features
#SBATCH -n 1
#SBATCH --cpus-per-task=8
#SBATCH --mem=128G
#SBATCH -t 24:00:00
#SBATCH --error=/mnt/tank/scratch/rislamova/SRR_files/Logs/calc_features_%j.err
#SBATCH --output=/mnt/tank/scratch/rislamova/SRR_files/Logs/calc_features_%j.out

cd /mnt/tank/scratch/ris/SRR_files
source /nfs/home/ris/miniforge3/etc/profile.d/mamba.sh
mamba activate snakemake

metafx calc_features \
    -t 8 \
    -m 16G \
    -w wd_calc_features \
    -k 31 \
    -d wd_unique_bmd \
    -i $(cat files_arth_filtered.txt)

**Substep 3.3.2** Preprocess arthritis features

In [ ]:
import pandas as pd
import joblib

# Load model to get feature names
model = joblib.load("wd_fit/rf_model.joblib")
trained_features = model.feature_names_in_

# Load arthritis features
arth = pd.read_csv("wd_calc_features_arth/feature_table.tsv", sep="\t", index_col=0).T
arth.index = arth.index.str.replace("_r1.fastq.gz", "", regex=False).str.replace("_r1", "", regex=False)

# Keep only features used in training
arth_filt = arth[trained_features]
arth_norm = arth_filt.div(arth_filt.sum(axis=1), axis=0).fillna(0)

# Predict
preds = model.predict(arth_norm)
result = pd.DataFrame({"sample": arth_norm.index, "predicted": preds})
result.to_csv("predictions_arth.csv", index=False)
print("Predictions saved. Class distribution:\n", result["predicted"].value_counts())


**Substep 3.3.3** 5‑fold cross‑validation with metafx cv

In [ ]:
#!/bin/bash
#SBATCH -J cv_metafx
#SBATCH -n 1
#SBATCH --cpus-per-task=16
#SBATCH --mem=64G
#SBATCH -t 4:00:00

cd /mnt/tank/scratch/rislamova/SRR_files
source /nfs/home/rislamova/miniforge3/etc/profile.d/mamba.sh
mamba activate snakemake

metafx cv -t 16 -w wd_cv -f wd_unique_bmd/feature_table.tsv -i wd_unique_bmd/samples_categories.tsv -n 5 --grid

# The output accuracy was 0.894 ± 0.038